# 03 - Modelo de Recomendación
## Sistema de recomendación de películas basado en similitud de coseno

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
load_dotenv('../.env')
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
print("API Key cargada:", "Sí" if TMDB_API_KEY else "No")

API Key cargada: Sí


### 3.1 Cargar datos

In [3]:
df = pd.read_csv('../data/processed/movies_clean.csv')
print(f"Películas cargadas: {len(df)}")
df.head()

Películas cargadas: 193


,date_rated,title,year,letterboxd_url,my_score
0,2026-04-10,The Marvels,2023,https://boxd.it/mwU4,1.5
1,2026-04-10,Spider-Man,2002,https://boxd.it/2a8i,3.5
2,2026-04-10,Spider-Man 3,2007,https://boxd.it/2a7Y,2.5
3,2026-04-10,Spider-Man 2,2004,https://boxd.it/2a88,4.5
4,2026-04-10,Spider-Man: Into the Spider-Verse,2018,https://boxd.it/azpY,4.5


### 3.2 Obtener géneros desde la API de TMDb
Primero buscamos el ID de TMDb de cada película, luego obtenemos sus géneros.

In [4]:
def search_movie(title, year):
    try:
        url = "https://api.themoviedb.org/3/search/movie"
        params = {
            'api_key': TMDB_API_KEY,
            'query': title,
            'year': year
        }
        response = requests.get(url, params=params)
        if response.status_code == 200:
            results = response.json().get('results', [])
            if results:
                return results[0]['id']
        return None
    except:
        return None

def get_genres(tmdb_id):
    try:
        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
        params = {'api_key': TMDB_API_KEY}
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            return [g['name'] for g in data.get('genres', [])]
        return []
    except:
        return []

In [6]:
# Prueba con una película
test_title = df.iloc[1]['title']
test_year = df.iloc[1]['year']
test_id = search_movie(test_title, test_year)
print(f"Probando con: {test_title} ({test_year})")
print(f"TMDb ID: {test_id}")
print(f"Géneros: {get_genres(test_id)}")

Probando con: Spider-Man (2002)
TMDb ID: 557
Géneros: ['Action', 'Science Fiction']


### 3.3 Obtener géneros de todas las películas

In [7]:
tmdb_ids = {}
genres_dict = {}
total = len(df)

for i, row in df.iterrows():
    title = row['title']
    year = row['year']
    
    tmdb_id = search_movie(title, year)
    time.sleep(0.25)
    
    if tmdb_id:
        tmdb_ids[i] = tmdb_id
        genres = get_genres(tmdb_id)
        time.sleep(0.25)
        genres_dict[i] = genres
    else:
        genres_dict[i] = []
    
    if (i + 1) % 10 == 0:
        print(f"Progreso: {i + 1}/{total} - Último: {title} -> {genres_dict[i]}")

found = sum(1 for g in genres_dict.values() if g)
print(f"\nCompletado. Géneros encontrados para {found}/{total} películas.")

Progreso: 10/193 - Último: The Amazing Spider-Man 2 -> ['Action', 'Adventure', 'Science Fiction']
Progreso: 20/193 - Último: Coraline -> ['Animation', 'Family', 'Fantasy']
Progreso: 30/193 - Último: Guardians of the Galaxy -> ['Action', 'Science Fiction', 'Adventure']
Progreso: 40/193 - Último: Avengers: Age of Ultron -> ['Action', 'Adventure', 'Science Fiction']
Progreso: 50/193 - Último: Don't Look Up -> ['Comedy', 'Science Fiction', 'Drama']
Progreso: 60/193 - Último: Dawn of the Planet of the Apes -> ['Science Fiction', 'Action', 'Drama', 'Thriller']
Progreso: 70/193 - Último: Independence Day -> ['Action', 'Adventure', 'Science Fiction']
Progreso: 80/193 - Último: Hulk -> ['Science Fiction', 'Adventure', 'Action']
Progreso: 90/193 - Último: Shin Godzilla -> ['Action', 'Science Fiction', 'Horror']
Progreso: 100/193 - Último: Evangelion: 2.0 You Can (Not) Advance -> ['Animation', 'Science Fiction', 'Action', 'Drama']
Progreso: 110/193 - Último: Kung Fu Panda -> ['Animation', 'Family

### 3.4 Agregar géneros al DataFrame y guardar

In [8]:
df['genres'] = df.index.map(genres_dict)
df['genres_str'] = df['genres'].apply(lambda x: ', '.join(x) if x else 'Unknown')

# Filtrar películas sin géneros
df = df[df['genres'].apply(len) > 0].copy()
df.reset_index(drop=True, inplace=True)

df.to_csv('../data/processed/movies_with_genres.csv', index=False)

print("Géneros únicos encontrados:")
all_genres = set(g for genres in df['genres'] for g in genres)
print(sorted(all_genres))
print(f"\nTotal de géneros únicos: {len(all_genres)}")
print(f"Películas con géneros: {len(df)}")
df[['title', 'my_score', 'genres_str']].head(10)

Géneros únicos encontrados:
['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War']

Total de géneros únicos: 16
Películas con géneros: 183


,title,my_score,genres_str
0,The Marvels,1.5,"Science Fiction, Adventure, Action"
1,Spider-Man,3.5,"Action, Science Fiction"
2,Spider-Man 3,2.5,"Action, Adventure, Science Fiction"
3,Spider-Man 2,4.5,"Action, Adventure, Science Fiction"
4,Spider-Man: Into the Spider-Verse,4.5,"Animation, Action, Adventure, Science Fiction"
5,Spider-Man: Across the Spider-Verse,4.5,"Animation, Action, Adventure, Science Fiction"
6,Spider-Man: No Way Home,3.5,"Action, Adventure, Science Fiction"
7,Spider-Man: Homecoming,1.5,"Action, Adventure, Science Fiction"
8,Spider-Man: Far From Home,1.5,"Action, Adventure, Science Fiction"
9,The Amazing Spider-Man 2,4.5,"Action, Adventure, Science Fiction"


### 3.5 Crear matriz de características (One-Hot Encoding)

In [9]:
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genres'])
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=df['title'])

print(f"Forma de la matriz: {genre_df.shape}")
print(f"(cada película es un vector de {genre_df.shape[1]} dimensiones)\n")
genre_df.head()

Forma de la matriz: (183, 16)
(cada película es un vector de 16 dimensiones)



,Action,Adventure,Animation,Comedy,Crime,Drama,Family,Fantasy,History,Horror,Mystery,Romance,Science Fiction,TV Movie,Thriller,War
title,,,,,,,,,,,,,,,,
The Marvels,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
Spider-Man,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
Spider-Man 3,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
Spider-Man 2,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
Spider-Man: Into the Spider-Verse,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0


### 3.6 Construir perfil de gustos del usuario
Ponderamos los géneros por la calificación del usuario (escala 0.5 - 5.0).

In [10]:
weights = df['my_score'].values.reshape(-1, 1)
weighted_genres = genre_matrix * weights
user_profile = weighted_genres.sum(axis=0) / weights.sum()

user_profile_df = pd.Series(user_profile, index=mlb.classes_).sort_values(ascending=False)
print("Tu perfil de gustos (todos los géneros):\n")
print(user_profile_df)

Tu perfil de gustos (todos los géneros):

Action             0.639535
Science Fiction    0.524313
Animation          0.522199
Adventure          0.514799
Fantasy            0.352008
Drama              0.241015
Family             0.169133
Comedy             0.168076
Romance            0.075053
Horror             0.063425
Thriller           0.060254
Mystery            0.043340
Crime              0.020085
War                0.009514
History            0.004228
TV Movie           0.004228
dtype: float64


### 3.7 Buscar películas nuevas para recomendar
Usamos la API de TMDb para buscar películas populares y bien calificadas por género.

In [11]:
# IDs de géneros en TMDb
genre_response = requests.get(
    "https://api.themoviedb.org/3/genre/movie/list",
    params={'api_key': TMDB_API_KEY}
)
tmdb_genre_ids = {g['name']: g['id'] for g in genre_response.json()['genres']}

# Top 5 géneros del usuario
top_genres = user_profile_df.head(5).index.tolist()
print(f"Tus top 5 géneros: {top_genres}\n")

# Buscar candidatos
my_tmdb_ids = set(tmdb_ids.values())
nuevas = []

for genre_name in top_genres:
    if genre_name not in tmdb_genre_ids:
        continue
    gid = tmdb_genre_ids[genre_name]
    
    for page in [1, 2]:
        url = "https://api.themoviedb.org/3/discover/movie"
        params = {
            'api_key': TMDB_API_KEY,
            'with_genres': gid,
            'sort_by': 'vote_average.desc',
            'vote_count.gte': 500,
            'page': page
        }
        try:
            response = requests.get(url, params=params)
            time.sleep(0.25)
            if response.status_code == 200:
                for movie in response.json().get('results', []):
                    if movie['id'] not in my_tmdb_ids:
                        genres = get_genres(movie['id'])
                        time.sleep(0.25)
                        nuevas.append({
                            'tmdb_id': movie['id'],
                            'title': movie['title'],
                            'year': movie.get('release_date', '')[:4],
                            'score_tmdb': movie.get('vote_average', 0),
                            'genres': genres
                        })
        except:
            continue

df_nuevas = pd.DataFrame(nuevas).drop_duplicates(subset='tmdb_id')
print(f"Películas candidatas encontradas: {len(df_nuevas)}")

Tus top 5 géneros: ['Action', 'Science Fiction', 'Animation', 'Adventure', 'Fantasy']

Películas candidatas encontradas: 82


### 3.8 Calcular similitud entre el perfil del usuario y las películas candidatas

In [12]:
nuevas_matrix = mlb.transform(df_nuevas['genres'])
similitudes = cosine_similarity([user_profile], nuevas_matrix)[0]

df_nuevas['similitud'] = similitudes
df_nuevas['genres_str'] = df_nuevas['genres'].apply(lambda x: ', '.join(x))

recomendaciones = df_nuevas.sort_values('similitud', ascending=False).head(20)
recomendaciones[['title', 'year', 'score_tmdb', 'similitud', 'genres_str']].reset_index(drop=True)

/home/edgar/Projects/movie-recommender/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['Music'] will be ignored
  warnings.warn(


,title,year,score_tmdb,similitud,genres_str
0,My Hero Academia: Heroes Rising,2019,8.130,0.834807,"Animation, Action, Fantasy, Adventure"
1,Akira,1988,7.900,0.801201,"Animation, Science Fiction, Action"
2,Star Wars,1977,8.202,0.797684,"Adventure, Action, Science Fiction"
3,Inception,2010,8.372,0.797684,"Action, Science Fiction, Adventure"
4,Return of the Jedi,1983,7.906,0.797684,"Adventure, Action, Science Fiction"
5,The Empire Strikes Back,1980,8.393,0.797684,"Adventure, Action, Science Fiction"
6,Mortal Kombat Legends: Scorpion's Revenge,2020,8.088,0.719322,"Animation, Action, Fantasy"
7,New Gods: Nezha Reborn,2021,8.054,0.719322,"Animation, Fantasy, Action"
8,Jujutsu Kaisen 0,2021,8.110,0.719322,"Animation, Action, Fantasy"
9,The Lord of the Rings: The Return of the King,2003,8.496,0.715806,"Adventure, Fantasy, Action"


In [13]:
recomendaciones[['title', 'year', 'score_tmdb', 'similitud', 'genres_str']].to_csv(
    '../output/recomendaciones_peliculas.csv', index=False
)
print("Recomendaciones guardadas en output/recomendaciones_peliculas.csv")

Recomendaciones guardadas en output/recomendaciones_peliculas.csv
